In [1]:
import numpy as np

from sklearn.metrics import confusion_matrix
from transformers import (
    Trainer,
    DataCollatorForTokenClassification,
    TrainingArguments
)

def confusion_matrix_manual(preds, labels):
    TP = FP = TN = FN = 0

    for true_seq, pred_seq in zip(labels, preds):
        true_seq = list(true_seq)
        pred_seq = list(pred_seq)
        true_has_answer = any(label == 1 for label in true_seq)
        pred_has_answer = any(label == 1 for label in pred_seq)

        if true_has_answer and pred_has_answer:
            if true_seq == pred_seq:
                TP += 1
            else:
                FN += 1  # mismatched spans count as wrong
        elif true_has_answer and not pred_has_answer:
            FN += 1
        elif not true_has_answer and pred_has_answer:
            FP += 1
        else:
            TN += 1

    return TP, FN, TN, FP

def compute_metrics_eval(eval_pred, answerable_flags):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=2)
    
    pred_answerable = [True if 1 in seq else False for seq in preds]

    # Mask out ignored tokens (-100)
    mask = labels != -100
    preds = [p[m] for p, m in zip(preds, mask)]
    labels = [l[m] for l, m in zip(labels, mask)]
    
    TP, FN, TN, FP = confusion_matrix_manual(preds, labels)
    
    total = TP + FN + TN + FP
    accuracy = (TP + TN) / total if total > 0 else 0.0
    
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # Answerable True False Scores
    cm = confusion_matrix(answerable_flags, pred_answerable, labels=[True, False])
    
    # Accuracy per answerable class
    # Accuracy = TP / (TP + FN) for each class
    acc_true = cm[0, 0] / cm[0].sum() if cm[0].sum() > 0 else 0.0
    acc_false = cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0.0
    cm = cm.flatten().tolist()

    return {
        "accuracy": accuracy,
        "f1": f1,
        "accuracy_answerable_true": acc_true,
        "accuracy_answerable_false": acc_false,
        "cm": cm
    }
    
def evaluate_binary(model, tokenizer, test_set, evaluation="All"):
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    answerable_flags = np.array(test_set["answerable"], dtype=bool)

    # Trainer only for evaluation
    trainer = Trainer(
        model=model,
        eval_dataset=test_set,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=lambda eval_pred: compute_metrics_eval(eval_pred, answerable_flags),
    )

    results = trainer.evaluate()

    # Log metrics nicely
    print(f"Evaluation Results {evaluation}:")
    for k, v in results.items():
        if k.startswith("eval_"):
            print(f"{k}: {v}")

    return results


/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments, Trainer
)
import numpy as np

# 1. Load the dataset and select the three languages
raw_datasets = load_dataset("coastalcph/tydi_xor_rc")
languages = ["ko", "te", "ar"]

train_datasets = {
    lang: raw_datasets["train"].filter(lambda x, l=lang: x["lang"] == l)
    for lang in languages
}
val_datasets = {
    lang: raw_datasets["validation"].filter(lambda x, l=lang: x["lang"] == l)
    for lang in languages
}

# 2. Choose a multilingual model
checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess_function(examples):
    """Convert each (question, context) pair into input_ids, attention_mask and per-token labels."""
    questions = [q.strip() for q in examples["question"]]
    # Tokenize and get offset mappings
    encoding = tokenizer(
        questions,
        examples["context"],
        max_length=512,
        truncation="only_second",
        return_offsets_mapping=True,
        padding="max_length",
    )
    offset_mapping = encoding.pop("offset_mapping")
    labels = []

    for i, offsets in enumerate(offset_mapping):
        # Determine the answer span; if unanswerable, start_char/end_char remain None
        if not examples["answerable"][i] or examples["answer_start"][i] == -1:
            start_char = end_char = None
        else:
            start_char = examples["answer_start"][i]
            end_char   = start_char + len(examples["answer"][i])

        sequence_ids = encoding.sequence_ids(i)
        label = []
        for seq_id, (start, end) in zip(sequence_ids, offsets):
            if seq_id != 1:
                # token belongs to the question or a special token → ignore in loss
                label.append(-100)
            else:
                if start_char is None:
                    # unanswerable → all context tokens are 0
                    label.append(0)
                else:
                    # mark tokens whose span overlaps the answer as 1, else 0
                    is_answer = (start < end_char) and (end > start_char)
                    label.append(1 if is_answer else 0)
        labels.append(label)

    encoding["labels"] = labels
    return encoding

def make_compute_metrics(answerable_flags):
    """
    Returns a compute_metrics function for HF Trainer that:
      - Keeps token-level precision/recall/F1/accuracy
      - Adds token-acc on answerable/unanswerable subsets
      - Adds question-level accuracy on answerable/unanswerable
    `answerable_flags` must be a 1D bool/0-1 array aligned with eval_dataset order.
    """
    ans_flags = np.asarray(answerable_flags, dtype=bool)

    def compute_metrics(eval_pred):
        predictions, labels = eval_pred  # predictions: [N, L, C], labels: [N, L]
        if isinstance(predictions, tuple):  # some HF versions return (logits, ...)
            predictions = predictions[0]
        preds = np.argmax(predictions, axis=-1)  # [N, L]

        # --- Global token-level confusion counts (ignoring -100) ---
        TP = FP = FN = TN = 0
        token_acc_per_example = []   # per-example token accuracy
        q_correct_ans = []           # question-level correct for answerable examples
        q_correct_un  = []           # question-level correct for unanswerable examples

        N = labels.shape[0]
        for i in range(N):
            mask = (labels[i] != -100)          # only context tokens (you set others to -100)
            if not np.any(mask):
                # edge case: no valid tokens; skip from per-example stats
                continue

            true_i = labels[i][mask]            # 0/1
            pred_i = preds[i][mask]             # 0/1

            TP += np.sum((true_i == 1) & (pred_i == 1))
            FP += np.sum((true_i == 0) & (pred_i == 1))
            FN += np.sum((true_i == 1) & (pred_i == 0))
            TN += np.sum((true_i == 0) & (pred_i == 0))

            # Per-example token accuracy
            token_acc_per_example.append(np.mean(true_i == pred_i))

            # Question-level correctness
            if ans_flags[i]:
                # Correct if predicted at least one of the true answer tokens
                # (overlap of predicted=1 with gold=1)
                q_correct_ans.append(bool(np.any((true_i == 1) & (pred_i == 1))))
            else:
                # Correct if predicted no answer tokens at all
                q_correct_un.append(not bool(np.any(pred_i == 1)))

        precision = TP / (TP + FP + 1e-8)
        recall    = TP / (TP + FN + 1e-8)
        f1        = 2 * precision * recall / (precision + recall + 1e-8)
        accuracy  = (TP + TN) / (TP + TN + FP + FN + 1e-8)

        token_acc_per_example = np.array(token_acc_per_example, dtype=float)

        # Build masks over examples (length N) for token-acc splits
        mask_ans = ans_flags[:N]
        mask_un  = ~ans_flags[:N]

        # To align with token_acc_per_example length, we recompute per-example token acc only
        # for examples that had any valid tokens; build a parallel list of indices we counted:
        counted_idx = []
        for i in range(N):
            if np.any(labels[i] != -100):
                counted_idx.append(i)
        counted_idx = np.array(counted_idx, dtype=int)

        # Split token accuracy by answerable/unanswerable on counted examples
        if counted_idx.size > 0:
            ans_mask_counted = mask_ans[counted_idx]
            un_mask_counted  = mask_un[counted_idx]
            token_acc_ans = float(token_acc_per_example[ans_mask_counted].mean()) if np.any(ans_mask_counted) else float("nan")
            token_acc_un  = float(token_acc_per_example[un_mask_counted].mean())  if np.any(un_mask_counted)  else float("nan")
        else:
            token_acc_ans = token_acc_un = float("nan")

        # Question-level accuracies
        q_acc_ans = float(np.mean(q_correct_ans)) if len(q_correct_ans) > 0 else float("nan")
        q_acc_un  = float(np.mean(q_correct_un))  if len(q_correct_un)  > 0 else float("nan")

        return {
            # Original token-level metrics
            "precision": precision,
            "recall":    recall,
            "f1":        f1,
            "accuracy":  accuracy,
            # New: token-accuracy by question type
            "token_acc_answerable":   token_acc_ans,
            "token_acc_unanswerable": token_acc_un,
            # New: question-level accuracy by question type
            "q_acc_answerable":   q_acc_ans,
            "q_acc_unanswerable": q_acc_un,
        }

    return compute_metrics

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# 3. Preprocess the data
overall_train_set = concatenate_datasets(list(train_datasets.values()))
tokenized_train = overall_train_set.map(
    preprocess_function,
    batched=True,
    remove_columns=overall_train_set.column_names,
)
overall_val_set = concatenate_datasets(list(val_datasets.values()))
tokenized_val = overall_val_set.map(
    preprocess_function,
    batched=True,
    remove_columns=overall_val_set.column_names,
)

# 4. Load a fresh model for this language
model = AutoModelForTokenClassification.from_pretrained(checkpoint, num_labels=2)

# 5. Set up training arguments; adjust as needed for your resources
training_args = TrainingArguments(
    output_dir=f"tydi_token_classifier",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    report_to = []
)

# 6. Train and evaluate
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_train,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(np.array(overall_train_set["answerable"], dtype=bool)),
)

trainer.train()

evaluate_binary(model, tokenizer, overall_val_set)

results = {}
for lang in languages:
    tokenized_val = val_datasets[lang].map(
        preprocess_function,
        batched=True,
        remove_columns=val_datasets[lang].column_names,
    )
    metrics = trainer.evaluate(tokenized_val)
    results[lang] = metrics
    print(f"Validation metrics for {lang}: {metrics}")
    
    evaluate_binary(model, tokenizer, tokenized_val, lang)

print("Final results:", results)


Map: 100%|██████████| 1155/1155 [00:00<00:00, 4461.22 examples/s]
Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/rc/g6hbpj9x7fzfptshgk1vqn9m0000gp/T/ipykernel_18416/1653425935.py:198: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/xk84vl/Documents/Repos/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 